# Notebook 1 — RoBERTa-large-BNE + MLP potente

Clasificación de década con RoBERTa grande, MLP potente, accuracy como métrica principal, patience=2, submissions normal y full-data.

In [ ]:
# =========================
# Instalación del ambiente
# =========================
# Ejecuta esta celda al inicio del notebook. Después de instalar, reinicia el kernel.

%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
%pip install transformers accelerate sentencepiece pandas numpy scikit-learn matplotlib tqdm scipy


In [ ]:
# =========================
# Imports y configuración base
# =========================

import os
import gc
import math
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 70)
print("CONFIGURACIÓN DEL ENTORNO")
print("=" * 70)
print("Python OK")
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA PyTorch:", torch.version.cuda)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria GPU:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("=" * 70)


In [ ]:
# =========================
# Limpieza segura de memoria
# =========================

def clean_memory(extra_vars=None):
    extra_vars = extra_vars or []
    for name in extra_vars:
        if name in globals():
            try:
                del globals()[name]
            except Exception:
                pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("GPU asignada:", round(torch.cuda.memory_allocated() / 1024**3, 3), "GB")
        print("GPU reservada:", round(torch.cuda.memory_reserved() / 1024**3, 3), "GB")

clean_memory()


## 1. Configuración del experimento

Usamos RoBERTa grande en español, MLP potente, métrica principal `accuracy`, y `patience=2`.

In [ ]:
# =========================
# Configuración
# =========================

TRAIN_PATH = "train.csv"
EVAL_PATH = "eval.csv"

TEXT_COL = "text"
LABEL_COL = "decade"

MODEL_NAME = "PlanTL-GOB-ES/roberta-large-bne"

MAX_LEN = 512
MAX_TEXT_CHARS = 3000

EPOCHS = 30
PATIENCE = 2

BATCH_SIZE = 16 if DEVICE.type == "cuda" else 4
GRAD_ACCUM_STEPS = 1

LR_ROBERTA = 1e-5
LR_MLP = 2e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.08
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0

MLP_HIDDEN_1 = 2048
MLP_HIDDEN_2 = 1024
MLP_HIDDEN_3 = 512
DROPOUT = 0.35

CHECKPOINT_DIR = Path("checkpoints_roberta_large_mlp")
CHECKPOINT_DIR.mkdir(exist_ok=True)

BEST_MODEL_PATH = CHECKPOINT_DIR / "best_roberta_large_mlp.pt"
LAST_MODEL_PATH = CHECKPOINT_DIR / "last_roberta_large_mlp.pt"

print("Modelo:", MODEL_NAME)
print("Batch size:", BATCH_SIZE)
print("Max len:", MAX_LEN)
print("Patience:", PATIENCE)


## 2. Carga de datos y mapeo de etiquetas

El mapeo se revisa explícitamente para evitar errores con las clases extremas `150` y `188`.

In [ ]:
# =========================
# Cargar train.csv
# =========================

df = pd.read_csv(TRAIN_PATH)

assert TEXT_COL in df.columns, f"Falta columna {TEXT_COL}"
assert LABEL_COL in df.columns, f"Falta columna {LABEL_COL}"

df = df[[TEXT_COL, LABEL_COL]].copy()
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df[df[TEXT_COL].str.len() > 0].reset_index(drop=True)

decades = sorted(df[LABEL_COL].unique())
decade_to_idx = {decade: idx for idx, decade in enumerate(decades)}
idx_to_decade = {idx: decade for decade, idx in decade_to_idx.items()}

df["label_idx"] = df[LABEL_COL].map(decade_to_idx)

NUM_CLASSES = len(decades)

assert decades[0] == 150, f"La primera década debería ser 150, pero es {decades[0]}"
assert decades[-1] == 188, f"La última década debería ser 188, pero es {decades[-1]}"
assert NUM_CLASSES == 39, f"Esperábamos 39 clases, pero hay {NUM_CLASSES}"
assert decade_to_idx[150] == 0
assert decade_to_idx[188] == 38
assert idx_to_decade[0] == 150
assert idx_to_decade[38] == 188

print("Shape:", df.shape)
print("Clases:", NUM_CLASSES)
print("Mapping extremo:", {150: decade_to_idx[150], 188: decade_to_idx[188]})
display(df.head())
display(df[LABEL_COL].value_counts().sort_index())


In [ ]:
# =========================
# Split train / val / test
# =========================

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label_idx"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_idx"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = part[LABEL_COL].value_counts().sort_index()
    print(name, "150:", counts.get(150, 0), "188:", counts.get(188, 0))


## 3. Tokenizer, datasets y DataLoaders

In [ ]:
# =========================
# Tokenizer
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)


In [ ]:
# =========================
# Crops de texto
# =========================

def crop_train_text(text, max_chars=MAX_TEXT_CHARS):
    text = str(text)
    if len(text) <= max_chars:
        return text
    start = random.randint(0, len(text) - max_chars)
    return text[start:start + max_chars]


def crop_eval_text(text, max_chars=MAX_TEXT_CHARS, mode="center"):
    text = str(text)
    if len(text) <= max_chars:
        return text

    if mode == "start":
        return text[:max_chars]
    if mode == "end":
        return text[-max_chars:]

    start = (len(text) - max_chars) // 2
    return text[start:start + max_chars]


def get_eval_crops(text, max_chars=MAX_TEXT_CHARS):
    text = str(text)
    if len(text) <= max_chars:
        return [text]

    crops = [
        text[:max_chars],
        text[(len(text) - max_chars) // 2:(len(text) - max_chars) // 2 + max_chars],
        text[-max_chars:],
    ]

    out, seen = [], set()
    for c in crops:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out


In [ ]:
# =========================
# Dataset RoBERTa
# =========================

class TextDataset(Dataset):
    def __init__(self, dataframe, train_mode=False):
        self.texts = dataframe[TEXT_COL].astype(str).tolist()
        self.labels = dataframe["label_idx"].astype(int).tolist()
        self.train_mode = train_mode

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        if self.train_mode:
            text = crop_train_text(text)
        else:
            text = crop_eval_text(text, mode="center")

        return {
            "text": text,
            "label": self.labels[idx],
        }


def collate_text_batch(batch):
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)

    encoded = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": labels,
        "texts": texts,
    }


train_dataset = TextDataset(train_df, train_mode=True)
val_dataset = TextDataset(val_df, train_mode=False)
test_dataset = TextDataset(test_df, train_mode=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_text_batch, num_workers=0, pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_text_batch, num_workers=0, pin_memory=(DEVICE.type == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_text_batch, num_workers=0, pin_memory=(DEVICE.type == "cuda"))

print("Batches train:", len(train_loader))
print("Batches val:", len(val_loader))
print("Batches test:", len(test_loader))


## 4. Modelo RoBERTa grande + MLP potente

In [ ]:
# =========================
# Modelo RoBERTa + MLP
# =========================

class RobertaLargeMLPClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()

        self.roberta = AutoModel.from_pretrained(model_name)
        hidden = self.roberta.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden, MLP_HIDDEN_1),
            nn.LayerNorm(MLP_HIDDEN_1),
            nn.GELU(),
            nn.Dropout(DROPOUT),

            nn.Linear(MLP_HIDDEN_1, MLP_HIDDEN_2),
            nn.LayerNorm(MLP_HIDDEN_2),
            nn.GELU(),
            nn.Dropout(DROPOUT),

            nn.Linear(MLP_HIDDEN_2, MLP_HIDDEN_3),
            nn.LayerNorm(MLP_HIDDEN_3),
            nn.GELU(),
            nn.Dropout(DROPOUT),

            nn.Linear(MLP_HIDDEN_3, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls)
        return logits


model = RobertaLargeMLPClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)

try:
    model.roberta.gradient_checkpointing_enable()
    print("Gradient checkpointing activado.")
except Exception as e:
    print("No se pudo activar gradient checkpointing:", e)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parámetros totales: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")


In [ ]:
# =========================
# Forward pass de prueba
# =========================

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

model.eval()
with torch.no_grad():
    with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
        logits = model(input_ids, attention_mask)

print("input_ids:", input_ids.shape)
print("logits:", logits.shape)
print("labels:", labels.shape)
assert logits.shape == (labels.size(0), NUM_CLASSES)

clean_memory(["input_ids", "attention_mask", "labels", "logits", "batch"])


## 5. Entrenamiento con accuracy como métrica principal

In [ ]:
# =========================
# Métricas
# =========================

idx_to_decade_array = np.array([idx_to_decade[i] for i in range(NUM_CLASSES)])

def compute_metrics_np(y_true_idx, y_pred_idx):
    y_true_idx = np.asarray(y_true_idx)
    y_pred_idx = np.asarray(y_pred_idx)

    true_decades = idx_to_decade_array[y_true_idx]
    pred_decades = idx_to_decade_array[y_pred_idx]

    abs_err = np.abs(true_decades - pred_decades)

    return {
        "acc": float(np.mean(y_true_idx == y_pred_idx)),
        "mae": float(np.mean(abs_err)),
        "acc_pm1": float(np.mean(abs_err <= 1)),
        "acc_pm2": float(np.mean(abs_err <= 2)),
    }


In [ ]:
# =========================
# Optimizer, scheduler y AMP
# =========================

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

roberta_params = []
head_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if name.startswith("roberta."):
        roberta_params.append(param)
    else:
        head_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": roberta_params, "lr": LR_ROBERTA, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": LR_MLP, "weight_decay": WEIGHT_DECAY},
    ]
)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_training_steps = num_update_steps_per_epoch * EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

print("Optimizer listo.")
print("Steps:", total_training_steps, "Warmup:", warmup_steps)


In [ ]:
# =========================
# Train / evaluate
# =========================

def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    total_examples = 0
    all_preds = []
    all_labels = []

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, desc="Training", leave=False), start=1):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss_for_backward = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_for_backward).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_examples += bs

        preds = torch.argmax(logits.detach(), dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

    metrics = compute_metrics_np(all_labels, all_preds)
    metrics["loss"] = total_loss / total_examples
    return metrics


@torch.no_grad()
def evaluate(model, loader, desc="Evaluating"):
    model.eval()

    total_loss = 0.0
    total_examples = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, desc=desc, leave=False):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_examples += bs

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

    metrics = compute_metrics_np(all_labels, all_preds)
    metrics["loss"] = total_loss / total_examples
    metrics["preds"] = np.array(all_preds)
    metrics["labels"] = np.array(all_labels)
    return metrics


In [ ]:
# =========================
# Loop principal con patience=2
# =========================

history = []
best_val_acc = -1.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 80)

    train_metrics = train_one_epoch(model, train_loader)
    val_metrics = evaluate(model, val_loader, desc="Validation")

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_mae": train_metrics["mae"],
        "train_pm1": train_metrics["acc_pm1"],
        "train_pm2": train_metrics["acc_pm2"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_mae": val_metrics["mae"],
        "val_pm1": val_metrics["acc_pm1"],
        "val_pm2": val_metrics["acc_pm2"],
        "lr_roberta": optimizer.param_groups[0]["lr"],
        "lr_mlp": optimizer.param_groups[1]["lr"],
    }

    history.append(row)

    print(f"Train acc: {row['train_acc']:.4f} | Train loss: {row['train_loss']:.4f} | Train MAE: {row['train_mae']:.3f}")
    print(f"Val acc:   {row['val_acc']:.4f} | Val loss:   {row['val_loss']:.4f} | Val MAE:   {row['val_mae']:.3f}")
    print(f"Val ±1: {row['val_pm1']:.4f} | Val ±2: {row['val_pm2']:.4f}")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "history": history,
        "best_val_acc": best_val_acc,
        "decade_to_idx": decade_to_idx,
        "idx_to_decade": idx_to_decade,
        "decades": decades,
        "config": {
            "MODEL_NAME": MODEL_NAME,
            "MAX_LEN": MAX_LEN,
            "NUM_CLASSES": NUM_CLASSES,
            "MLP_HIDDEN_1": MLP_HIDDEN_1,
            "MLP_HIDDEN_2": MLP_HIDDEN_2,
            "MLP_HIDDEN_3": MLP_HIDDEN_3,
            "DROPOUT": DROPOUT,
        }
    }, LAST_MODEL_PATH)

    if val_metrics["acc"] > best_val_acc:
        best_val_acc = val_metrics["acc"]
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "best_val_acc": best_val_acc,
            "best_val_metrics": val_metrics,
            "decade_to_idx": decade_to_idx,
            "idx_to_decade": idx_to_decade,
            "decades": decades,
            "config": {
                "MODEL_NAME": MODEL_NAME,
                "MAX_LEN": MAX_LEN,
                "NUM_CLASSES": NUM_CLASSES,
                "MLP_HIDDEN_1": MLP_HIDDEN_1,
                "MLP_HIDDEN_2": MLP_HIDDEN_2,
                "MLP_HIDDEN_3": MLP_HIDDEN_3,
                "DROPOUT": DROPOUT,
            }
        }, BEST_MODEL_PATH)

        print("Nuevo mejor modelo guardado.")
    else:
        epochs_without_improvement += 1
        print(f"Sin mejora por {epochs_without_improvement} época(s).")

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping activado.")
        break

    clean_memory()

print("Mejor época:", best_epoch)
print("Mejor val accuracy:", best_val_acc)


## 6. Evaluación, matriz de confusión y submission normal

In [ ]:
# =========================
# Cargar mejor checkpoint en CPU y evaluar test
# =========================

clean_memory(["optimizer", "scheduler", "scaler"])

checkpoint = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Checkpoint:", BEST_MODEL_PATH)
print("Época:", checkpoint["epoch"])
print("Best val acc:", checkpoint["best_val_acc"])

test_metrics = evaluate(model, test_loader, desc="Test")

print("=" * 70)
print("TEST")
print("=" * 70)
print("Accuracy:", test_metrics["acc"])
print("MAE:", test_metrics["mae"])
print("±1:", test_metrics["acc_pm1"])
print("±2:", test_metrics["acc_pm2"])


In [ ]:
# =========================
# Matriz de confusión sin normalizar
# =========================

cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"], labels=list(range(NUM_CLASSES)))

cm_df = pd.DataFrame(
    cm,
    index=[idx_to_decade[i] for i in range(NUM_CLASSES)],
    columns=[idx_to_decade[i] for i in range(NUM_CLASSES)]
)

display(cm_df)

plt.figure(figsize=(15, 12))
plt.imshow(cm, aspect="auto")
plt.title("Matriz de confusión sin normalizar")
plt.xlabel("Década predicha")
plt.ylabel("Década real")
plt.xticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)], rotation=90)
plt.yticks(np.arange(NUM_CLASSES), [idx_to_decade[i] for i in range(NUM_CLASSES)])
plt.colorbar(label="Conteo")
plt.tight_layout()
plt.show()

print("Predicciones por clase en test:")
print(pd.Series([idx_to_decade[int(i)] for i in test_metrics["preds"]]).value_counts().sort_index())


In [ ]:
# =========================
# Multi-crop prediction
# =========================

@torch.no_grad()
def predict_proba_texts_multicrop(model, texts, batch_size=BATCH_SIZE):
    model.eval()
    all_probs = []

    for text in tqdm(texts, desc="Predicting multicrop"):
        crops = get_eval_crops(text)
        crop_probs = []

        for start in range(0, len(crops), batch_size):
            batch_crops = crops[start:start + batch_size]
            encoded = tokenizer(
                batch_crops,
                truncation=True,
                padding=True,
                max_length=MAX_LEN,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(DEVICE)
            attention_mask = encoded["attention_mask"].to(DEVICE)

            with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
                logits = model(input_ids, attention_mask)
                probs = torch.softmax(logits, dim=1)

            crop_probs.append(probs.detach().cpu())

        avg_probs = torch.cat(crop_probs, dim=0).mean(dim=0)
        all_probs.append(avg_probs.numpy())

    return np.vstack(all_probs)


def save_submission_from_probs(probs, ids, path):
    pred_idx = probs.argmax(axis=1)
    answers = [idx_to_decade[int(i)] for i in pred_idx]

    sub = pd.DataFrame({"id": ids, "answer": answers})
    sub.to_csv(path, index=False)

    print("Guardado:", path)
    display(sub.head())
    display(sub["answer"].value_counts().sort_index())
    return sub


In [ ]:
# =========================
# Submission normal con modelo validado
# =========================

SUBMISSION_PATH = "submission_roberta_large_mlp_validated.csv"

eval_df = pd.read_csv(EVAL_PATH)
assert "id" in eval_df.columns and "text" in eval_df.columns

eval_df["text"] = eval_df["text"].fillna("").astype(str)

eval_probs = predict_proba_texts_multicrop(model, eval_df["text"].tolist())
submission_validated = save_submission_from_probs(eval_probs, eval_df["id"].values, SUBMISSION_PATH)


## 7. Entrenamiento final con todo el dataset y submission full-data

In [ ]:
# =========================
# Configuración final full-data
# =========================

FINAL_EPOCHS = int(checkpoint["epoch"])

FINAL_CHECKPOINT_DIR = Path("checkpoints_roberta_large_mlp_full_data")
FINAL_CHECKPOINT_DIR.mkdir(exist_ok=True)

FINAL_MODEL_PATH = FINAL_CHECKPOINT_DIR / "final_roberta_large_mlp_full_data.pt"

print("FINAL_EPOCHS:", FINAL_EPOCHS)


In [ ]:
# =========================
# Dataset full-data
# =========================

full_df = df.copy().reset_index(drop=True)

full_dataset = TextDataset(full_df, train_mode=True)
full_loader = DataLoader(
    full_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_text_batch,
    num_workers=0,
    pin_memory=(DEVICE.type == "cuda")
)

print("Full examples:", len(full_dataset))
print("Full batches:", len(full_loader))


In [ ]:
# =========================
# Modelo final desde cero
# =========================

clean_memory(["model"])

final_model = RobertaLargeMLPClassifier(MODEL_NAME, NUM_CLASSES).to(DEVICE)

try:
    final_model.roberta.gradient_checkpointing_enable()
except Exception:
    pass

final_criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

roberta_params = []
head_params = []

for name, param in final_model.named_parameters():
    if name.startswith("roberta."):
        roberta_params.append(param)
    else:
        head_params.append(param)

final_optimizer = torch.optim.AdamW(
    [
        {"params": roberta_params, "lr": LR_ROBERTA, "weight_decay": WEIGHT_DECAY},
        {"params": head_params, "lr": LR_MLP, "weight_decay": WEIGHT_DECAY},
    ]
)

steps_per_epoch = math.ceil(len(full_loader) / GRAD_ACCUM_STEPS)
total_steps = steps_per_epoch * FINAL_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

final_scheduler = get_linear_schedule_with_warmup(
    final_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

final_scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

print("Modelo final listo.")


In [ ]:
# =========================
# Entrenamiento final
# =========================

def train_final_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_examples = 0
    all_preds, all_labels = [], []

    final_optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, desc="Final training", leave=False), start=1):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE.type == "cuda")):
            logits = model(input_ids, attention_mask)
            loss = final_criterion(logits, labels)
            loss_for_backward = loss / GRAD_ACCUM_STEPS

        final_scaler.scale(loss_for_backward).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            final_scaler.unscale_(final_optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            final_scaler.step(final_optimizer)
            final_scaler.update()
            final_optimizer.zero_grad(set_to_none=True)
            final_scheduler.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total_examples += bs
        all_preds.extend(torch.argmax(logits.detach(), dim=1).cpu().numpy().tolist())
        all_labels.extend(labels.detach().cpu().numpy().tolist())

    metrics = compute_metrics_np(all_labels, all_preds)
    metrics["loss"] = total_loss / total_examples
    return metrics


final_history = []

for epoch in range(1, FINAL_EPOCHS + 1):
    print(f"\nFinal epoch {epoch}/{FINAL_EPOCHS}")
    metrics = train_final_one_epoch(final_model, full_loader)
    final_history.append({"epoch": epoch, **metrics})
    print(metrics)
    clean_memory()

torch.save({
    "epoch": FINAL_EPOCHS,
    "model_state_dict": final_model.state_dict(),
    "history": final_history,
    "decade_to_idx": decade_to_idx,
    "idx_to_decade": idx_to_decade,
    "decades": decades,
    "config": {
        "MODEL_NAME": MODEL_NAME,
        "MAX_LEN": MAX_LEN,
        "NUM_CLASSES": NUM_CLASSES,
        "MLP_HIDDEN_1": MLP_HIDDEN_1,
        "MLP_HIDDEN_2": MLP_HIDDEN_2,
        "MLP_HIDDEN_3": MLP_HIDDEN_3,
        "DROPOUT": DROPOUT,
    }
}, FINAL_MODEL_PATH)

print("Modelo final guardado:", FINAL_MODEL_PATH)


In [ ]:
# =========================
# Submission full-data
# =========================

FINAL_SUBMISSION_PATH = "submission_roberta_large_mlp_full_data.csv"

final_probs = predict_proba_texts_multicrop(final_model, eval_df["text"].tolist())
submission_full = save_submission_from_probs(final_probs, eval_df["id"].values, FINAL_SUBMISSION_PATH)
